# 05 — Evaluation & Results

**Project #8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**  
Network Data Analysis Laboratory 2025-2026, Politecnico di Milano

---

## Goals
1. Load trained models and test-set arrays
2. Compute all performance metrics (MSE, RMSE, MAE, R², MAPE, training time)
3. Compare **Neural Network vs. Random Forest** across X values
4. Plot: predicted vs. true, residuals, feature importances, X sensitivity
5. Produce a clean comparison table for the report

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import torch
from pathlib import Path

from src.models.neural_network import MLPRegressor, predict_mlp, DEVICE
from src.models.random_forest  import RFRegressor
from src.utils.metrics import evaluate, print_metrics, measure_inference_time
from src.utils.visualization import (
    plot_pred_vs_true,
    plot_residuals,
    plot_metrics_comparison,
    plot_x_sensitivity,
    plot_feature_importance,
)

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../results/models')
FIGURES_DIR   = Path('../results/figures')

X_VALUES = [1, 3, 5, 10]
SAVE_FIGS = True

## 1. Load test data

In [ ]:
def load_test(prefix):
    return (
        np.load(f'{prefix}_X_test.npy'),
        np.load(f'{prefix}_y_test.npy'),
    )

X_test_base, y_test = load_test(PROCESSED_DIR / 'acc')
test_splits = {x: load_test(PROCESSED_DIR / f'acc_X{x}') for x in X_VALUES}

print(f"Baseline test: {X_test_base.shape}")
for x, (Xte, yte) in test_splits.items():
    print(f"X={x:2d} test: {Xte.shape}")

## 2. Load training results

In [ ]:
with open(MODELS_DIR / 'training_results.json') as f:
    training_results = json.load(f)

best_nn_params = training_results['best_nn_params']
best_hidden_dims = [best_nn_params[f'h{i}'] for i in range(best_nn_params['n_layers'])]

## 3. Load models and evaluate with inference timing

In [ ]:
all_metrics = {}   # {label: metrics_dict}

# --- Baseline (no neighbour features) ---
# RF baseline
rf_base = RFRegressor.load(MODELS_DIR / 'rf_acc_arena_base.pkl')
inf_rf = measure_inference_time(rf_base.predict, X_test_base)
preds = rf_base.predict(X_test_base)
m = evaluate(y_test, preds)
m['inference_time_ms'] = inf_rf
all_metrics['RF (X=0, baseline)'] = m

# MLP baseline
mlp_base = MLPRegressor(input_dim=X_test_base.shape[1], hidden_dims=best_hidden_dims,
                        dropout=best_nn_params['dropout'])
mlp_base.load_state_dict(torch.load(MODELS_DIR / 'mlp_acc_arena_tuned_base.pt',
                                    map_location=DEVICE))
inf_mlp = measure_inference_time(lambda x: predict_mlp(mlp_base, x), X_test_base)
preds = predict_mlp(mlp_base, X_test_base)
m = evaluate(y_test, preds)
m['inference_time_ms'] = inf_mlp
all_metrics['MLP (X=0, baseline)'] = m

# --- With neighbour features ---
for x_val in X_VALUES:
    X_te, y_te = test_splits[x_val]

    rf_x = RFRegressor.load(MODELS_DIR / f'rf_acc_arena_X{x_val}.pkl')
    inf_rf = measure_inference_time(rf_x.predict, X_te)
    preds_rf = rf_x.predict(X_te)
    m_rf = evaluate(y_te, preds_rf)
    m_rf['inference_time_ms'] = inf_rf
    all_metrics[f'RF  (X={x_val})'] = m_rf

    mlp_x = MLPRegressor(input_dim=X_te.shape[1], hidden_dims=best_hidden_dims,
                         dropout=best_nn_params['dropout'])
    mlp_x.load_state_dict(torch.load(MODELS_DIR / f'mlp_acc_arena_X{x_val}.pt',
                                     map_location=DEVICE))
    inf_mlp = measure_inference_time(lambda x: predict_mlp(mlp_x, x), X_te)
    preds_mlp = predict_mlp(mlp_x, X_te)
    m_mlp = evaluate(y_te, preds_mlp)
    m_mlp['inference_time_ms'] = inf_mlp
    all_metrics[f'MLP (X={x_val})'] = m_mlp

print('Evaluation complete.')

## 4. Results Table

In [ ]:
results_df = pd.DataFrame(all_metrics).T
results_df = results_df[['mse', 'rmse', 'mae', 'r2', 'mape_%',
                          'training_time_s', 'inference_time_ms']]
results_df = results_df.round(4)
print('=== Full Results Table ===')
display(results_df)

In [ ]:
# LaTeX-ready table
print(results_df.to_latex(float_format='%.4f'))

## 5. Metric Comparison Plots

In [ ]:
# Compare all models on RMSE, MAE, R²
fig = plot_metrics_comparison(all_metrics, metrics=['rmse', 'mae', 'r2'], save=SAVE_FIGS)
plt.show()

In [ ]:
# X sensitivity: RF
rf_metrics_per_x = [all_metrics[f'RF  (X={x})'] for x in X_VALUES]
fig = plot_x_sensitivity(X_VALUES, rf_metrics_per_x, metric='rmse',
                         model_name='Random Forest', save=SAVE_FIGS)
plt.show()

# X sensitivity: MLP
mlp_metrics_per_x = [all_metrics[f'MLP (X={x})'] for x in X_VALUES]
fig = plot_x_sensitivity(X_VALUES, mlp_metrics_per_x, metric='rmse',
                         model_name='MLP', save=SAVE_FIGS)
plt.show()

## 6. Best Model: Predictions vs. True

In [ ]:
# Identify best X for each model type
best_x_rf  = min(X_VALUES, key=lambda x: all_metrics[f'RF  (X={x})']['rmse'])
best_x_mlp = min(X_VALUES, key=lambda x: all_metrics[f'MLP (X={x})']['rmse'])
print(f"Best X for RF : {best_x_rf}")
print(f"Best X for MLP: {best_x_mlp}")

In [ ]:
# Predicted vs. True — best RF
X_te_best, y_te_best = test_splits[best_x_rf]
rf_best = RFRegressor.load(MODELS_DIR / f'rf_acc_arena_X{best_x_rf}.pkl')
preds_rf_best = rf_best.predict(X_te_best)

fig = plot_pred_vs_true(y_te_best, preds_rf_best,
                        model_name=f'RF (X={best_x_rf})', save=SAVE_FIGS)
plt.show()

fig = plot_residuals(y_te_best, preds_rf_best,
                     model_name=f'RF (X={best_x_rf})', save=SAVE_FIGS)
plt.show()

In [ ]:
# Predicted vs. True — best MLP
X_te_best_mlp, y_te_best_mlp = test_splits[best_x_mlp]
mlp_best = MLPRegressor(input_dim=X_te_best_mlp.shape[1], hidden_dims=best_hidden_dims,
                         dropout=best_nn_params['dropout'])
mlp_best.load_state_dict(torch.load(MODELS_DIR / f'mlp_acc_arena_X{best_x_mlp}.pt',
                                    map_location=DEVICE))
preds_mlp_best = predict_mlp(mlp_best, X_te_best_mlp)

fig = plot_pred_vs_true(y_te_best_mlp, preds_mlp_best,
                        model_name=f'MLP (X={best_x_mlp})', save=SAVE_FIGS)
plt.show()

fig = plot_residuals(y_te_best_mlp, preds_mlp_best,
                     model_name=f'MLP (X={best_x_mlp})', save=SAVE_FIGS)
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
import joblib
from src.data.preprocessor import Preprocessor
from src.features.closest_users import get_neighbor_column_names, NEIGHBOR_FEATURES
from src.data.preprocessor import NUMERIC_FEATURES

# Load preprocessor for best X to get feature names
prep = Preprocessor.load(PROCESSED_DIR / f'acc_X{best_x_rf}_preprocessor.pkl')
feature_names = prep.feature_names_out_

if feature_names and rf_best.feature_importances_ is not None:
    fig = plot_feature_importance(
        rf_best.feature_importances_,
        feature_names,
        top_n=20,
        model_name=f'RF (X={best_x_rf})',
        save=SAVE_FIGS,
    )
    plt.show()

## 8. Key Findings

> Fill in after running the notebook:

- **Best model overall**: …
- **Effect of X (closest users)**: …
- **NN vs RF comparison**: …
- **Most important features (RF)**: …
- **Training time vs. accuracy trade-off**: …